# 🏛️ CompliAssist AI – MSME Compliance Platform
### Lab Notebook | SEPM Project

**Project:** CompliAssist – An AI-powered compliance assistant for Indian MSMEs  
**Tech Stack:** Vite + React (Frontend) | Node.js HTTP Server (Backend) | Docker (Deployment)  
**Purpose of this notebook:** Data analysis, compliance risk modelling, loan recommendation scoring, and project metrics.

---

## Table of Contents
1. [Project Architecture Overview](#architecture)
2. [Seed Data Loading & Exploration](#data)
3. [Compliance Analysis & Risk Scoring](#compliance)
4. [Loan Recommendation Analysis](#loans)
5. [Government Scheme Matching](#schemes)
6. [Alert & Deadline Analysis](#alerts)
7. [AI Query Assistant – Pattern Analysis](#ai)
8. [Dashboard KPIs & Visualisation](#dashboard)
9. [API Endpoint Testing (via requests)](#api)
10. [Docker & Deployment Metrics](#docker)
11. [Project Metrics & SEPM Summary](#sepm)

---
## 1. Project Architecture Overview <a id='architecture'></a>

In [ ]:
# Install required libraries (run once)
# !pip install pandas matplotlib seaborn requests numpy plotly

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import json
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Libraries loaded successfully')

In [ ]:
# ─── Project Architecture Diagram (text-based) ───────────────────────────────
architecture = """
┌──────────────────────────────────────────────────────────────────────┐
│                        COMPLIASSIST PLATFORM                         │
├─────────────────────────┬────────────────────────────────────────────┤
│       FRONTEND          │               BACKEND                      │
│  Vite + React (SPA)     │  Node.js HTTP Server  (port 8787)          │
│                         │                                            │
│  Pages:                 │  API Routes:                               │
│  • Dashboard            │  POST /api/auth/login                      │
│  • MSME Profile         │  GET  /api/auth/session                    │
│  • Compliance Guidance  │  GET  /api/bootstrap                       │
│  • Govt Schemes         │  PUT  /api/profile                         │
│  • Loan Recommendations │  PUT  /api/settings                        │
│  • Document Storage     │  PUT  /api/guidance/documents/:id          │
│  • Alerts & Deadlines   │  POST /api/alerts/:id/action               │
│  • AI Query Assistant   │  POST /api/schemes/:id/apply               │
│  • Settings             │  POST /api/loans/compare                   │
│                         │  POST /api/loans/:id/apply                 │
│                         │  GET  /api/loans/:id/eligibility           │
│                         │  POST /api/assistant/query                 │
│                         │  POST /api/documents/upload                │
│                         │  GET  /api/documents/:id/download          │
│                         │  DELETE /api/documents/:id                 │
│                         │  GET  /api/health                          │
├─────────────────────────┴────────────────────────────────────────────┤
│            Data Layer: JSON flat-file store (backend/data/)          │
│            File Uploads: backend/uploads/  (Docker volume)           │
├──────────────────────────────────────────────────────────────────────┤
│                     Docker (multi-stage build)                       │
│  Stage 1 (build): node:20-alpine → npm ci → vite build              │
│  Stage 2 (runtime): node:20-alpine → production deps → node server  │
└──────────────────────────────────────────────────────────────────────┘
"""
print(architecture)

---
## 2. Seed Data Loading & Exploration <a id='data'></a>

In [ ]:
# ─── Seed data (mirrors backend/seedData.js) ─────────────────────────────────
seed_data = {
    "profile": {
        "companyName": "TechNova Solutions Pvt Ltd",
        "location": "Bengaluru, Karnataka",
        "businessType": "Private Limited Company",
        "industry": "Information Technology",
        "companySize": "11 - 50 Employees (Small)",
        "udyamRegistration": "UDYAM-KR-00-1234567",
        "employees": 32,
        "foundedYear": "2018",
        "pan": "AAACT1234F",
    },
    "compliances": [
        {"id": "cmp-gstr1",  "title": "GST Filing (GSTR-1)",           "date": "2026-03-15", "status": "PENDING"},
        {"id": "cmp-epf",   "title": "EPF Contribution",               "date": "2026-03-11", "status": "COMPLETED"},
        {"id": "cmp-tds",   "title": "TDS Payment",                    "date": "2026-03-18", "status": "PENDING"},
        {"id": "cmp-ptax",  "title": "Professional Tax Payment",       "date": "2026-03-08", "status": "OVERDUE"},
    ],
    "alerts": [
        {"id": "alert-ptax",     "type": "overdue",  "title": "Professional Tax Payment Overdue", "date": "2026-03-08", "status": "OPEN"},
        {"id": "alert-gstr1",    "type": "warning",  "title": "GST Filing (GSTR-1) Due Soon",    "date": "2026-03-15", "status": "OPEN"},
        {"id": "alert-guidance", "type": "info",     "title": "Updated Data Privacy Guidance",   "date": "2026-03-12", "status": "OPEN"},
    ],
    "schemes": [
        {"id": "scheme-cgs",     "title": "Credit Guarantee Scheme (CGS)", "ministry": "MSME Ministry",              "match": 96, "tags": ["FINANCING", "MANUFACTURING"], "status": "RECOMMENDED"},
        {"id": "scheme-digital", "title": "Digital MSME Scheme",           "ministry": "Ministry of Electronics and IT", "match": 91, "tags": ["TECHNOLOGY", "DIGITIZATION"],   "status": "RECOMMENDED"},
        {"id": "scheme-zed",     "title": "ZED Certification Scheme",      "ministry": "QCI and Ministry of MSME",  "match": 77, "tags": ["QUALITY", "SUSTAINABILITY"],    "status": "RECOMMENDED"},
    ],
    "loans": [
        {"id": "loan-hdfc",  "bank": "HDFC Bank",          "type": "MSME Growth Loan",      "interest": "8.45% - 10.50%", "minRate": 8.45, "amount": "Rs 10L - Rs 50L",  "maxAmountLakhs": 50,  "tenure": "Up to 5 years", "purpose": "Working Capital"},
        {"id": "loan-sidbi", "bank": "SIDBI",              "type": "SPEED Plus",            "interest": "6.75% onwards",  "minRate": 6.75, "amount": "Rs 25L - Rs 2Cr", "maxAmountLakhs": 200, "tenure": "Up to 3 years", "purpose": "Machinery Purchase"},
        {"id": "loan-sbi",   "bank": "State Bank of India","type": "SME Smart Score Loan",  "interest": "7.90% onwards",  "minRate": 7.90, "amount": "Rs 50L - Rs 3Cr", "maxAmountLakhs": 300, "tenure": "Up to 7 years", "purpose": "Expansion Capital"},
    ],
    "requiredDocs": [
        {"id": "doc-sales-register",  "name": "Sales Register",   "status": "ready"},
        {"id": "doc-bank-statement",  "name": "Bank Statement",    "status": "ready"},
        {"id": "doc-export-records",  "name": "Export Records",    "status": "pending"},
        {"id": "doc-previous-return", "name": "Previous Return",   "status": "ready"},
    ]
}

print("✅ Seed data loaded")
print(f"Company : {seed_data['profile']['companyName']}")
print(f"Industry: {seed_data['profile']['industry']}")
print(f"Location: {seed_data['profile']['location']}")
print(f"Employees: {seed_data['profile']['employees']}")

In [ ]:
# Convert to DataFrames
df_compliance = pd.DataFrame(seed_data['compliances'])
df_alerts     = pd.DataFrame(seed_data['alerts'])
df_schemes    = pd.DataFrame(seed_data['schemes'])
df_loans      = pd.DataFrame(seed_data['loans'])
df_docs       = pd.DataFrame(seed_data['requiredDocs'])

df_compliance['date'] = pd.to_datetime(df_compliance['date'])
df_alerts['date']     = pd.to_datetime(df_alerts['date'])

print("=== Compliance Table ===")
print(df_compliance[['title','date','status']].to_string(index=False))

---
## 3. Compliance Analysis & Risk Scoring <a id='compliance'></a>

In [ ]:
# ─── Compliance Health Score (mirrors backend store.js logic) ─────────────────
total      = len(df_compliance)
completed  = (df_compliance['status'] == 'COMPLETED').sum()
pending    = (df_compliance['status'] == 'PENDING').sum()
overdue    = (df_compliance['status'] == 'OVERDUE').sum()

health_score = round((completed / total) * 100)
health_label = ('Excellent' if health_score >= 85
                else 'Good' if health_score >= 70
                else 'Needs Attention')

print(f"Compliance Health Score : {health_score}%  →  {health_label}")
print(f"  ✅ Completed : {completed}")
print(f"  ⏳ Pending   : {pending}")
print(f"  🚨 Overdue   : {overdue}")

In [ ]:
# ─── Compliance Status Pie Chart ──────────────────────────────────────────────
status_counts = df_compliance['status'].value_counts()
colors = {'COMPLETED': '#22c55e', 'PENDING': '#f59e0b', 'OVERDUE': '#ef4444'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie
axes[0].pie(
    status_counts.values,
    labels=status_counts.index,
    colors=[colors[s] for s in status_counts.index],
    autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[0].set_title('Compliance Status Distribution', fontweight='bold', fontsize=13)

# Timeline bar
today = pd.Timestamp('2026-03-23')
df_compliance['days_until_due'] = (df_compliance['date'] - today).dt.days
bar_colors = [colors[s] for s in df_compliance['status']]
axes[1].barh(df_compliance['title'], df_compliance['days_until_due'], color=bar_colors)
axes[1].axvline(0, color='black', linewidth=1.2, linestyle='--', label='Today')
axes[1].set_xlabel('Days Until Due (negative = overdue)')
axes[1].set_title('Compliance Deadline Timeline', fontweight='bold', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.suptitle('CompliAssist – Compliance Dashboard', y=1.02, fontsize=15, fontweight='bold')
plt.savefig('compliance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as compliance_analysis.png')

In [ ]:
# ─── Risk Score Model ─────────────────────────────────────────────────────────
# Simple rule-based risk scoring for each compliance item

def compute_risk_score(row, today=pd.Timestamp('2026-03-23')):
    """Higher score = higher risk (0-100)"""
    score = 0
    if row['status'] == 'OVERDUE':
        score += 60
        days_late = (today - row['date']).days
        score += min(30, days_late * 2)   # up to 30 more for lateness
    elif row['status'] == 'PENDING':
        days_left = (row['date'] - today).days
        if days_left <= 3:
            score += 50
        elif days_left <= 7:
            score += 30
        else:
            score += 10
    # COMPLETED → score stays 0
    return min(score, 100)

df_compliance['risk_score'] = df_compliance.apply(compute_risk_score, axis=1)

print("=== Compliance Risk Scores ===")
print(df_compliance[['title', 'status', 'date', 'days_until_due', 'risk_score']]
      .sort_values('risk_score', ascending=False)
      .to_string(index=False))

In [ ]:
# Risk Score Bar Chart
fig, ax = plt.subplots(figsize=(10, 4))
risk_colors = ['#ef4444' if s >= 60 else '#f59e0b' if s >= 30 else '#22c55e'
               for s in df_compliance['risk_score']]
bars = ax.barh(df_compliance['title'], df_compliance['risk_score'], color=risk_colors)
ax.set_xlabel('Risk Score (0 = Safe, 100 = Critical)')
ax.set_title('Compliance Item Risk Scores', fontweight='bold', fontsize=13)
ax.set_xlim(0, 100)

for bar, score in zip(bars, df_compliance['risk_score']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score}', va='center', fontweight='bold')

red_p   = mpatches.Patch(color='#ef4444', label='Critical (≥60)')
amber_p = mpatches.Patch(color='#f59e0b', label='Warning (30-59)')
green_p = mpatches.Patch(color='#22c55e', label='Safe (<30)')
ax.legend(handles=[red_p, amber_p, green_p], loc='lower right')
plt.tight_layout()
plt.savefig('risk_scores.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Loan Recommendation Analysis <a id='loans'></a>

In [ ]:
print("=== Available Loan Options ===")
print(df_loans[['bank','type','interest','amount','tenure','purpose']].to_string(index=False))

In [ ]:
# ─── Loan Scoring Model ───────────────────────────────────────────────────────
# Score each loan for this MSME (IT sector, 32 employees, working-capital need)
def loan_score(row, purpose_need='Working Capital'):
    score = 0
    # Lower interest rate → higher score
    score += max(0, (10 - row['minRate']) * 10)   # max 32.5 pts at 6.75%
    # Higher max amount → better
    score += min(30, row['maxAmountLakhs'] / 10)
    # Purpose match
    if purpose_need.lower() in row['purpose'].lower():
        score += 20
    return round(score, 1)

df_loans['recommendation_score'] = df_loans.apply(loan_score, axis=1)
df_loans_sorted = df_loans.sort_values('recommendation_score', ascending=False)

print("=== Loan Recommendation Ranking ===")
print(df_loans_sorted[['bank','type','minRate','maxAmountLakhs','purpose','recommendation_score']]
      .to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Interest rate comparison
axes[0].bar(df_loans['bank'], df_loans['minRate'],
            color=['#6366f1','#06b6d4','#10b981'])
axes[0].set_ylabel('Min Interest Rate (%)')
axes[0].set_title('Loan Interest Rate Comparison', fontweight='bold')
for i, (rate, bank) in enumerate(zip(df_loans['minRate'], df_loans['bank'])):
    axes[0].text(i, rate + 0.1, f'{rate}%', ha='center', fontweight='bold')

# Recommendation score
axes[1].barh(df_loans_sorted['bank'], df_loans_sorted['recommendation_score'],
             color=['#6366f1','#06b6d4','#10b981'])
axes[1].set_xlabel('Recommendation Score')
axes[1].set_title('Loan Recommendation Ranking\n(for Working Capital need)', fontweight='bold')

plt.suptitle('CompliAssist – Loan Recommendation Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('loan_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Government Scheme Matching <a id='schemes'></a>

In [ ]:
print("=== Government Scheme Match Scores ===")
print(df_schemes[['title','ministry','match','status']].sort_values('match', ascending=False).to_string(index=False))

In [ ]:
# Simulate scheme scan logic (mirrors /api/schemes/scan)
def refresh_scheme_scores(schemes, industry='Information Technology'):
    result = []
    is_tech = 'technology' in industry.lower()
    for i, scheme in enumerate(schemes):
        bonus = 5 if is_tech and 'TECHNOLOGY' in scheme['tags'] else 0
        new_match = max(68, min(99, scheme['match'] + bonus - i))
        result.append({**scheme, 'match_after_scan': new_match})
    return pd.DataFrame(result)

df_schemes_scan = refresh_scheme_scores(seed_data['schemes'])
print("=== After Eligibility Scan ===")
print(df_schemes_scan[['title','match','match_after_scan']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(df_schemes_scan))
width = 0.35
b1 = ax.bar(x - width/2, df_schemes_scan['match'],           width, label='Before Scan', color='#94a3b8')
b2 = ax.bar(x + width/2, df_schemes_scan['match_after_scan'],width, label='After Scan',  color='#6366f1')
ax.set_xticks(x)
ax.set_xticklabels(df_schemes_scan['title'], rotation=15, ha='right')
ax.set_ylabel('Match Score (%)')
ax.set_ylim(60, 100)
ax.set_title('Scheme Eligibility Match – Before vs After Profile Scan', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('scheme_match.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Alert & Deadline Analysis <a id='alerts'></a>

In [ ]:
print("=== Active Alerts ===")
print(df_alerts[['title','type','date','status']].to_string(index=False))

# Alert breakdown
alert_type_counts = df_alerts['type'].value_counts()
print(f"\nOpen alerts: {(df_alerts['status']=='OPEN').sum()}")
print(f"Overdue alerts: {(df_alerts['type']=='overdue').sum()}")

fig, ax = plt.subplots(figsize=(6, 4))
alert_colors = {'overdue': '#ef4444', 'warning': '#f59e0b', 'info': '#3b82f6'}
ax.bar(alert_type_counts.index,
       alert_type_counts.values,
       color=[alert_colors.get(t, '#94a3b8') for t in alert_type_counts.index])
ax.set_title('Alert Types Distribution', fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('alert_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. AI Query Assistant – Pattern Analysis <a id='ai'></a>

In [ ]:
# ─── AI Assistant Query Categories ───────────────────────────────────────────
# The backend uses keyword-based routing for the AI assistant
# Here we simulate and analyse query category coverage

query_categories = {
    'Greetings':          ['hi', 'hello', 'hey', 'namaste'],
    'GST / GSTR-1':       ['gst', 'gstr-1', 'gstr1', 'gst filing'],
    'EPF / PF':           ['epf', 'provident fund', 'pf'],
    'TDS':                ['tds', 'tax deducted', 'tcs'],
    'Professional Tax':   ['professional tax', 'ptax'],
    'Loans / Finance':    ['loan', 'finance', 'credit', 'borrow', 'fund'],
    'Govt Schemes':       ['scheme', 'subsidy', 'grant', 'ministry'],
    'Documents':          ['document', 'upload', 'vault', 'file', 'storage'],
    'Alerts / Deadlines': ['alert', 'deadline', 'overdue', 'urgent', 'pending'],
    'Compliance Status':  ['compliance', 'status', 'health', 'score'],
    'Company Profile':    ['profile', 'company', 'about', 'business'],
    'Udyam':              ['udyam', 'registration', 'msme certificate'],
    'Settings':           ['setting', 'notification', 'email', 'remind'],
    'Income Tax':         ['income tax', 'itr', 'it return'],
    'Dashboard':          ['dashboard', 'summary', 'overview'],
}

cat_df = pd.DataFrame({
    'Category': list(query_categories.keys()),
    'Keyword_Count': [len(v) for v in query_categories.values()]
})

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(cat_df['Category'], cat_df['Keyword_Count'], color='#6366f1')
ax.set_xlabel('Number of Trigger Keywords')
ax.set_title('AI Query Assistant – Query Category Coverage', fontweight='bold', fontsize=13)
for bar, cnt in zip(bars, cat_df['Keyword_Count']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            str(cnt), va='center')
plt.tight_layout()
plt.savefig('ai_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Total intent categories: {len(query_categories)}")
print(f"Total trigger keywords : {sum(len(v) for v in query_categories.values())}")

In [ ]:
# ─── Simulate the AI assistant locally ───────────────────────────────────────
def simulate_assistant_category(question):
    q = question.lower()
    for category, keywords in query_categories.items():
        if any(kw in q for kw in keywords):
            return category
    return 'Fallback / General'

test_queries = [
    "What is my GST filing status?",
    "Show available loans for my company",
    "Are there any overdue alerts?",
    "Which government schemes match my profile?",
    "What is my EPF contribution status?",
    "Hello!",
    "Explain TDS for my company",
    "What documents should I upload?",
    "Give me a dashboard summary",
]

print(f"{'Query':<45} {'Detected Category'}")
print('-' * 70)
for q in test_queries:
    print(f"{q:<45} {simulate_assistant_category(q)}")

---
## 8. Dashboard KPIs & Visualisation <a id='dashboard'></a>

In [ ]:
# ─── Full Dashboard KPI Summary ───────────────────────────────────────────────
kpis = {
    'Compliance Score':     f'{health_score}% ({health_label})',
    'Completed':            completed,
    'Pending':              pending,
    'Overdue':              overdue,
    'Open Alerts':          (df_alerts['status'] == 'OPEN').sum(),
    'Matched Schemes':      len(df_schemes),
    'Loan Options':         len(df_loans),
    'Required Docs Ready':  (df_docs['status'] == 'ready').sum(),
    'Required Docs Pending':(df_docs['status'] == 'pending').sum(),
    'Best Loan Rate':       f"{df_loans['minRate'].min()}% ({df_loans.loc[df_loans['minRate'].idxmin(), 'bank']})",
}

print("\n╔══════════════════════════════════════════════════╗")
print("║      COMPLIASSIST DASHBOARD KPIs                 ║")
print("╠══════════════════════════════════════════════════╣")
for k, v in kpis.items():
    print(f"║  {k:<30} {str(v):<16} ║")
print("╚══════════════════════════════════════════════════╝")

In [ ]:
# ─── Full Dashboard Canvas ────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
fig.suptitle('CompliAssist – Full Dashboard Visualisation', fontsize=16, fontweight='bold', y=1.01)

gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# 1. Compliance status pie
ax1 = fig.add_subplot(gs[0, 0])
status_counts = df_compliance['status'].value_counts()
ax1.pie(status_counts.values,
        labels=status_counts.index,
        colors=[colors[s] for s in status_counts.index],
        autopct='%1.0f%%', startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2))
ax1.set_title('Compliance Status', fontweight='bold')

# 2. Risk scores
ax2 = fig.add_subplot(gs[0, 1])
ax2.barh(df_compliance['title'], df_compliance['risk_score'],
         color=['#ef4444' if s >= 60 else '#f59e0b' if s >= 30 else '#22c55e'
                for s in df_compliance['risk_score']])
ax2.set_xlabel('Risk Score')
ax2.set_title('Compliance Risk Scores', fontweight='bold')
ax2.set_xlim(0, 100)

# 3. Scheme match scores
ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(range(len(df_schemes)), df_schemes['match'], color='#6366f1')
ax3.set_xticks(range(len(df_schemes)))
ax3.set_xticklabels([s['title'].split('(')[0].strip() for s in seed_data['schemes']], rotation=15, ha='right', fontsize=8)
ax3.set_ylabel('Match %')
ax3.set_ylim(60, 100)
ax3.set_title('Govt Scheme Match Scores', fontweight='bold')

# 4. Loan interest rates
ax4 = fig.add_subplot(gs[1, 0])
ax4.bar(df_loans['bank'], df_loans['minRate'], color=['#f59e0b','#10b981','#3b82f6'])
ax4.set_ylabel('Min Rate (%)')
ax4.set_title('Loan Interest Rates', fontweight='bold')
ax4.set_ylim(5, 11)
for i, r in enumerate(df_loans['minRate']):
    ax4.text(i, r + 0.1, f'{r}%', ha='center', fontsize=9, fontweight='bold')

# 5. Document checklist
ax5 = fig.add_subplot(gs[1, 1])
doc_status = df_docs['status'].value_counts()
ax5.bar(doc_status.index, doc_status.values,
        color=['#22c55e' if s=='ready' else '#ef4444' for s in doc_status.index])
ax5.set_title('Required Documents Status', fontweight='bold')
ax5.set_ylabel('Count')

# 6. Alert types
ax6 = fig.add_subplot(gs[1, 2])
at = df_alerts['type'].value_counts()
ax6.bar(at.index, at.values,
        color=[alert_colors.get(t, '#94a3b8') for t in at.index])
ax6.set_title('Alert Type Breakdown', fontweight='bold')
ax6.set_ylabel('Count')

plt.savefig('dashboard_full.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Full dashboard canvas saved')

---
## 9. API Endpoint Testing <a id='api'></a>

> **Pre-requisite:** Run the backend server locally first:  
> ```bash
> npm run build
> npm run start
> ```
> The server will be available at `http://localhost:8787`

In [ ]:
import requests

BASE_URL = 'http://localhost:8787'
TOKEN = None

def test_endpoint(method, path, payload=None, token=None, label=''):
    headers = {'Content-Type': 'application/json'}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    try:
        resp = requests.request(method, BASE_URL + path, json=payload, headers=headers, timeout=5)
        status_icon = '✅' if resp.status_code < 300 else '❌'
        print(f"{status_icon} [{resp.status_code}] {method:6} {path:<40} {label}")
        return resp
    except Exception as e:
        print(f"⚠️  Could not connect: {e}")
        return None

# 1. Health check
test_endpoint('GET', '/api/health', label='Health check')

In [ ]:
# 2. Login
resp = test_endpoint('POST', '/api/auth/login',
                     {'email': 'admin@technova.com', 'password': 'demo123'},
                     label='Login')
if resp and resp.status_code == 200:
    TOKEN = resp.json().get('token')
    print(f"   Token: {TOKEN[:16]}...")

In [ ]:
# 3. Authenticated endpoints
endpoints = [
    ('GET',  '/api/bootstrap',                 None,                         'Load app data'),
    ('GET',  '/api/auth/session',              None,                         'Session check'),
    ('POST', '/api/schemes/scan',              None,                         'Scheme eligibility scan'),
    ('POST', '/api/loans/compare',             None,                         'Loan comparison'),
    ('POST', '/api/assistant/query',           {'question': 'What is my GST status?'}, 'AI query'),
    ('GET',  '/api/loans/loan-sidbi/eligibility', None,                      'Loan eligibility check'),
]

for method, path, body, label in endpoints:
    test_endpoint(method, path, body, TOKEN, label)

---
## 10. Docker & Deployment Metrics <a id='docker'></a>

In [ ]:
# ─── Dockerfile stage analysis ────────────────────────────────────────────────
dockerfile_info = {
    'Stages': 2,
    'Base Image': 'node:20-alpine',
    'Build Stage Commands': ['FROM', 'WORKDIR', 'COPY (package.json)', 'RUN npm ci', 'COPY .', 'RUN npm run build'],
    'Runtime Stage Commands': ['FROM', 'WORKDIR', 'ENV (NODE_ENV, PORT, HOST)', 'COPY package.json',
                               'RUN npm ci --omit=dev', 'COPY dist', 'COPY backend/*.js',
                               'EXPOSE 8787', 'HEALTHCHECK', 'CMD'],
    'Exposed Port': 8787,
    'Healthcheck': 'GET /api/health every 30s',
    'Data Volumes': ['backend/data/', 'backend/uploads/'],
    'Excluded (dockerignore)': ['node_modules', 'dist', 'backend/data', 'backend/uploads', '.git'],
}

print("╔══════════════════════════════════════════════╗")
print("║      DOCKER DEPLOYMENT SUMMARY               ║")
print("╠══════════════════════════════════════════════╣")
for k, v in dockerfile_info.items():
    if isinstance(v, list):
        print(f"║  {k}:")
        for item in v:
            print(f"║    • {item}")
    else:
        print(f"║  {k:<25}: {v}")
print("╚══════════════════════════════════════════════╝")

print("""
Build command:
  docker build -t compliassist .

Run with persistent data:
  docker run -p 8787:8787 \\
    -v compliassist_data:/app/backend/data \\
    -v compliassist_uploads:/app/backend/uploads \\
    compliassist
""")

---
## 11. Project Metrics & SEPM Summary <a id='sepm'></a>

In [ ]:
import os

# Count source files and lines (run from the project root)
project_root = r'c:\Users\srijan\Desktop\sepm msme'

file_stats = {}
extensions = ['.jsx', '.js', '.css', '.html', '.json']

for root, dirs, files in os.walk(project_root):
    # Skip node_modules, dist, .git
    dirs[:] = [d for d in dirs if d not in ['node_modules', 'dist', '.git', 'uploads', 'data']]
    for fname in files:
        ext = os.path.splitext(fname)[1].lower()
        if ext in extensions:
            fpath = os.path.join(root, fname)
            try:
                with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
                    lines = len(f.readlines())
                file_stats[ext] = file_stats.get(ext, {'files': 0, 'lines': 0})
                file_stats[ext]['files'] += 1
                file_stats[ext]['lines'] += lines
            except:
                pass

df_stats = pd.DataFrame([
    {'Extension': ext, 'Files': v['files'], 'Lines of Code': v['lines']}
    for ext, v in file_stats.items()
]).sort_values('Lines of Code', ascending=False)

total_files = df_stats['Files'].sum()
total_lines = df_stats['Lines of Code'].sum()

print("=== Source Code Metrics ===")
print(df_stats.to_string(index=False))
print(f"\nTotal files : {total_files}")
print(f"Total LoC   : {total_lines}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_ext = {'#6366f1': '.jsx', '#06b6d4': '.js', '#10b981': '.css',
              '#f59e0b': '.html', '#ef4444': '.json'}
ext_colors = ['#6366f1','#06b6d4','#10b981','#f59e0b','#ef4444']

axes[0].bar(df_stats['Extension'], df_stats['Files'],
            color=ext_colors[:len(df_stats)])
axes[0].set_title('Files per Type', fontweight='bold')
axes[0].set_ylabel('Number of Files')

axes[1].bar(df_stats['Extension'], df_stats['Lines of Code'],
            color=ext_colors[:len(df_stats)])
axes[1].set_title('Lines of Code per Type', fontweight='bold')
axes[1].set_ylabel('Lines of Code')

plt.suptitle('CompliAssist – Source Code Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('code_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── SEPM Project Summary ─────────────────────────────────────────────────────
sepm_summary = {
    'Project Name':          'CompliAssist – MSME Compliance Platform',
    'Type':                  'Full-Stack Web Application',
    'Frontend':              'Vite + React 19 (SPA)',
    'Backend':               'Node.js HTTP server (no framework)',
    'Data Storage':          'Flat-file JSON store with seed data',
    'Authentication':        'Token-based sessions (UUID tokens)',
    'Deployment':            'Docker (multi-stage build, node:20-alpine)',
    'Pages Implemented':     10,
    'API Endpoints':         15,
    'AI Query Categories':   len(query_categories),
    'Compliance Items':      len(df_compliance),
    'Loan Options':          len(df_loans),
    'Govt Schemes':          len(df_schemes),
}

print("\n╔══════════════════════════════════════════════════════════════╗")
print("║                 SEPM PROJECT SUMMARY                        ║")
print("╠══════════════════════════════════════════════════════════════╣")
for k, v in sepm_summary.items():
    print(f"║  {k:<30} {str(v):<28} ║")
print("╚══════════════════════════════════════════════════════════════╝")